# Correlation between heart rate variability & reported user immersion in an VR experience

* Lucas Friborg Mitchell - lmitch21@student.aau.dk - Study No. 20213721


## Introduction

Quantifying a qualitative experience has always been a challenge. Using physiological data can help bridge the gap.

This mini-project was chosen to support a main semester project. As such, the data-set and processing will be mostly the same. This paper will however go more into depth on how the data was processed.

The semester project goal is to see if their is a correlation between task performance and level of immersion. An immersive VR experience is created, where the player has to make and serve cocktails for costumers. The game is split up into 3 different versions of increasing visual and sensory fidelity. 

The experiment is a between-subjects design, where each participant takes a 5 minute baseline measurement, followed by 1 of the 3 fidelity conditions. They then have 8 minutes to complete as many orders as possible.

## Implementation 

The heart data was acquired, using the Blood Volume Pulse (BVP) finger clip sensor from Plux Biosignals. As such, the data was streamed from OpenSignals and recorded with [LabRecorder](https://github.com/labstreaminglayer/App-LabRecorder). This allowed for simple management of multiple data streams and synchronization.

In [19]:
from scipy.signal import butter, filtfilt, find_peaks
import neurokit2 as nk
import numpy as np
import pyxdf
import matplotlib.pyplot as plt
import pandas as pd
from biosppy.signals.ppg import ppg
import heartpy as hp
import glob
import re
import os
from scipy import stats
from scipy.stats import shapiro, levene, f_oneway, kruskal, mannwhitneyu
from scipy.stats import f_oneway
from itertools import combinations
import seaborn as sns
from factor_analyzer import FactorAnalyzer
import pingouin as pg
import os

debugging = False
def Trace(message):
    """
    Function to print messages with a specific format.
    """
    if (debugging):
        print(f"[Trace] {message}")
    else:
        pass


In [20]:
def get_gaze_data(filepath = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-13/sub-13_ses-13_task-Baseline/_.xdf"):
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None
    gaze_stream = next((s for s in streams if s["info"]["name"][0] == "GazePointStream"), None)
    if gaze_stream:
        gaze_data = gaze_stream['time_series']
        gaze_ts = gaze_stream['time_stamps']
        return gaze_data, gaze_ts
    else:
        print(f"Warning: Gaze stream not found in {filepath}.")
        return None, None

gaze_data, gaze_ts = get_gaze_data()
# convert to DataFrame with time stamps in one axis and the object being looked at in the other axis
gaze_df = pd.DataFrame({'time': gaze_ts, 'layer': [l[0] for l in gaze_data]})
# print each unique layer
print(gaze_df["layer"].unique())

['0']


### Signal extraction
The first step is extracting the data stream from the file. The .XDF file contains dictionaries, formatted as XML. So we define a path to the specific dictionary and save the signal stream and time series to a list each.

In [21]:
def extract_bvp_and_markers_from_xdf(filepath):
    #print(f"Attempting to load XDF: {filepath}")
    try:
        streams, header = pyxdf.load_xdf(filepath)
    except Exception as e:
        print(f"Error loading XDF file {filepath}: {e}")
        return None, None, pd.DataFrame(), {'bvp': False, 'markers': False}
    bvp_s = next((s for s in streams if s["info"]["name"][0] == "OpenSignals" and any(ch['label'][0] == 'BVP0' for ch in s['info']['desc'][0]['channels'][0]['channel'])), None)
    mark_s = next((s for s in streams if s["info"]["name"][0] == "UnityMarkers"), None)
    streams_found = {'bvp': False, 'markers': False}
    bvp_signal, bvp_ts = None, None
    df_events = pd.DataFrame()
    if bvp_s:
        bvp_data_raw = bvp_s['time_series']
        bvp_ts_raw = bvp_s['time_stamps']
        bvp_channel_idx = next((i for i, ch in enumerate(bvp_s['info']['desc'][0]['channels'][0]['channel']) if ch['label'][0] == 'BVP0'), None)
        if bvp_channel_idx is not None and bvp_data_raw.ndim == 2 and bvp_data_raw.shape[1] > bvp_channel_idx:
            bvp_signal = bvp_data_raw[:, bvp_channel_idx].astype(np.float64)
            bvp_ts = bvp_ts_raw
            streams_found['bvp'] = True
            Trace(f"BVP stream found and BVP0 channel extracted from {filepath}.")
        else:
            print(f"Warning: BVP0 channel not found or data format unexpected in OpenSignals stream for {filepath}.")
    else:
        print(f"Warning: BVP stream (OpenSignals with BVP0) not found in {filepath}.")
    if mark_s:
        markers_raw = mark_s['time_series']
        marker_ts_raw = mark_s['time_stamps']
        if len(markers_raw) > 0:
            df_events = pd.DataFrame({'time': marker_ts_raw, 'event': [m[0] for m in markers_raw]})
            streams_found['markers'] = True
            #print(f"UnityMarkers stream found in {filepath}.")
        #else:
            #print(f"Warning: UnityMarkers stream found but no event data in {filepath}.")
    return bvp_signal, bvp_ts, df_events, streams_found


### Butterworth Bandpass Filter

Next, is filtering the raw data with a Butterworth filter [(Butterworth, S. (1930))](https://www.changpuak.ch/electronics/downloads/On_the_Theory_of_Filter_Amplifiers.pdf). It is commonly used for filtering ECG signals, as it helps remove high frequency noise. But it also works well for other types of signals to reduce noise.
The equation for the for a Butterworth Filter is given by:
$$
    G(w) = \frac{1}{\sqrt{1+w^2n}}
$$

Where $w$ is the angular frequency in radiants per second and $n$ is the number of poles in the filter. This can be modified into a bandpass filter to look like: 
$$
    G_{BP}(w) = \frac{1}{\sqrt{1+(\frac{B(w^2-w_0^2)}{ww_0})^{2n}}}
$$
Here $w_0 = \sqrt{w_Lw_H}$ as the low and high frequency cutoffs, and $B = w_H - w_L$ being the bandwidth.

The sample rate is based on the Nyquist frequency [(Shannon, Claude E. (1949))](https://doi.org/10.1109%2Fjrproc.1949.232969). Since we don't expect to see a heart rate of more than 200 BPM, the chosen sample rate will be 400 Hz.

In [22]:
def filter_bvp(signal, lowcut=0.5, highcut=8.0, fs=400):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    if high <= low:
        print(f"Warning: Highcut frequency ({highcut}Hz) is not above lowcut frequency ({lowcut}Hz) at fs={fs}Hz. Adjusting filter or skipping.")
        return signal
    try:
        b, a = butter(4, [low, high], btype='bandpass')
        filtered = filtfilt(b, a, signal)
        # plot before and after filtering for debugging
        if debugging:
            plt.figure(figsize=(12, 6))
            plt.subplot(2, 1, 1)
            plt.plot(signal, label='Original Signal')
            plt.title('Original Signal')
            plt.subplot(2, 1, 2)
            plt.plot(filtered, label='Filtered Signal', color='orange')
            plt.title('Filtered Signal')
            plt.tight_layout()
            plt.show()
        return filtered
    except ValueError as ve:
        print(f"ValueError during filtering: {ve}. Returning unfiltered signal.")
        return signal

### Calculating Heart Rate Variability

[HeartPy](https://pypi.org/project/heartpy/) is used to get a list of the peaks in the signal. The `hp.process()` function also further filters out very high and low heart rates, from 40 - 180 BPM.
Then we get the time difference interval between each peak (Inter-Beat Interval). Using the guidelines outlined by [(Camm et al., 1996)](doi.org/10.1093/oxfordjournals.eurheartj.a014868) we can then calculate the HRV in the form of Root Mean Square of Successive Differences (RMSSD).

In [23]:
def get_hrv_metrics(bvp_segment, timestamps, fs=400):
    if len(timestamps) != len(np.unique(timestamps)):
        print(f"Warning: The input 'timestamps' (bvp_ts_raw) array itself contains duplicate values. Number of timestamps: {len(timestamps)}, Unique timestamps: {len(np.unique(timestamps))}")
    if len(bvp_segment) < fs * 10:
        print(f"Warning: Segment too short ({len(bvp_segment)/fs:.2f}s, need at least 10s), skipping HRV.")
        return pd.Series(dtype=float)
    try:
        filtered_segment = filter_bvp(bvp_segment, fs=fs)
        wd, m = hp.process(filtered_segment, sample_rate=fs, calc_freq=False, high_precision=True, clean_rr=True)
        peaks = wd.get('peaklist', [])
        if len(peaks) < 5:
            print(f"Warning: Not enough peaks found ({len(peaks)}, need at least 5), skipping HRV.")
            return pd.Series(dtype=float)
        # Ensure peak indices are integers and within bounds before indexing timestamps
        valid_peaks = []
        for p in peaks:
            try:
                p_int = int(p) # Convert to integer
                if 0 <= p_int < len(timestamps): # Check bounds
                    valid_peaks.append(p_int)
            except (ValueError, TypeError):
                print(f"Warning: Invalid peak value {p} encountered, skipping it.")

        if len(valid_peaks) < 2:
            print(f"Warning: Not enough valid peaks ({len(valid_peaks)}) after filtering and conversion. Original peaks count from heartpy: {len(peaks)}. Skipping HRV.")
            return pd.Series(dtype=float)
        peak_times_sec = timestamps[valid_peaks]
        
        # debugging time differences
        time_diffs = np.diff(peak_times_sec)
        #print(f"Time differences between BVP samples: {time_diffs}")
        Trace(f"Mean time difference: {np.mean(time_diffs)}")
        Trace(f"Standard deviation of time differences: {np.std(time_diffs)}")
        
        ibi_ms = np.diff(peak_times_sec) * 1000
        if len(ibi_ms) < 3:
            print(f"Warning: Not enough IBIs calculated ({len(ibi_ms)}), skipping HRV.")
            return pd.Series(dtype=float)
        ibi_event_times_sec = peak_times_sec[1:]
        # debugging IBI to find dublicates
        Trace(f"unique ibi event times: {len(np.unique(ibi_event_times_sec))}")
        
        hrv_indices = nk.hrv({'RRI': ibi_ms, 'RRI_Time': ibi_event_times_sec}, sampling_rate=1000)
        return hrv_indices.iloc[0]
    except Exception as e:
        print(f"Error processing segment: {e}")
        return pd.Series(dtype=float)


### Running the functions

Below is the loop responsible for looping through the data files and grouping the different conditions. It ends by saving a data frame with each valid subject, the condition they tested, the phase, and HRV data.

In [32]:
def process_subject_data(subject_id, baseline_filepath, task_filepath, nominal_srate=400):
    results_list = []
    condition_from_task_folder = "Unknown"  # Default

    # --- Get Condition from Task File's PARENT FOLDER (if it exists) ---
    if task_filepath:
        task_parent_folder_name = os.path.basename(os.path.dirname(task_filepath))
        match_cond = re.search(r'_task-([^_\s]+)', task_parent_folder_name) # More robust regex for condition
        if match_cond:
            parsed_condition = match_cond.group(1)
            # Ensure 'Baseline' from folder name doesn't become the experimental condition
            if parsed_condition.lower() != 'baseline':
                condition_from_task_folder = parsed_condition
            else:
                print(f"Warning: Task filepath {task_filepath} seems to be from a folder named '...task-Baseline...'. Experimental condition remains 'Unknown' or will be based on a non-baseline task folder if available.")
        else:
            print(f"Warning: Could not parse condition from task folder name: {task_parent_folder_name} for subject {subject_id}")

    # --- 1. Process Baseline File ---
    if baseline_filepath:
        #print(f"Processing BASELINE for subject {subject_id} (Experimental Condition: {condition_from_task_folder}) from: {os.path.basename(baseline_filepath)}")
        bvp_baseline, ts_baseline, df_events_baseline, streams_baseline = extract_bvp_and_markers_from_xdf(baseline_filepath)

        if streams_baseline['bvp'] and bvp_baseline is not None and ts_baseline is not None:
            #print(f"Calculating baseline HRV for {subject_id} (full file)...")
            hrv_baseline_metrics = get_hrv_metrics(bvp_baseline, ts_baseline, fs=nominal_srate)
            if not hrv_baseline_metrics.empty:
                baseline_res = {'ParticipantID': subject_id, 'Condition': condition_from_task_folder, 'Phase': 'Baseline'}
                baseline_res.update(hrv_baseline_metrics)
                results_list.append(baseline_res)
                print(f"Baseline HRV calculated for {subject_id}.")
            else:
                print(f"No HRV metrics obtained for baseline for subject {subject_id}.")
        else:
            print(f"Could not process BVP for baseline for subject {subject_id} from {os.path.basename(baseline_filepath)}.")
    else:
        print(f"No baseline filepath provided for subject {subject_id}.")

    # --- 2. Process Task File ---
    if task_filepath:
        #print(f"Processing TASK for subject {subject_id}, Condition: {condition_from_task_folder} from: {os.path.basename(task_filepath)}")
        bvp_task_full, ts_task_full, df_events_task, streams_task = extract_bvp_and_markers_from_xdf(task_filepath)

        if streams_task['bvp'] and bvp_task_full is not None and ts_task_full is not None:
            #print(f"Calculating task HRV for {subject_id} (full file {len(bvp_task_full)/nominal_srate:.2f}s)...")
            hrv_task_metrics = get_hrv_metrics(bvp_task_full, ts_task_full, fs=nominal_srate)
            if not hrv_task_metrics.empty:
                task_res = {'ParticipantID': subject_id, 'Condition': condition_from_task_folder, 'Phase': 'Task'}
                task_res.update(hrv_task_metrics)
                results_list.append(task_res)
                print(f"Task HRV calculated for {subject_id}.")
            else:
                print(f"No HRV metrics obtained for task phase for subject {subject_id}.")
        else:
            print(f"Could not process BVP for task for subject {subject_id} from {os.path.basename(task_filepath)}.")
    else:
        print(f"No task filepath provided for subject {subject_id}.")

    return pd.DataFrame(results_list) if results_list else pd.DataFrame()

# --- Main Processing Loop ---
all_participant_dfs = [] 
data_root_folder = os.path.join(os.getcwd(), "Test_Data")
nominal_srate_main = 400

session_folders = [f.path for f in os.scandir(data_root_folder) if f.is_dir() and re.match(r'ses-\d+', f.name)]
Trace(f"Found session folders: {session_folders}")

for session_folder_path in session_folders:
    session_name = os.path.basename(session_folder_path)
    Trace(f"\nProcessing session: {session_name}")

    potential_st_folders = [f.path for f in os.scandir(session_folder_path) if f.is_dir() and "_task-" in f.name and "sub-" in f.name]

    subject_folders_map = {}
    for st_folder_path in potential_st_folders:
        st_folder_name = os.path.basename(st_folder_path)
        match_sub = re.search(r'(sub-[^_\s]+)', st_folder_name)
        if match_sub:
            subject_id_key = match_sub.group(1)
            if subject_id_key not in subject_folders_map:
                subject_folders_map[subject_id_key] = []
            subject_folders_map[subject_id_key].append(st_folder_path)
        else:
            print(f"  Warning: Could not parse subject ID from folder name {st_folder_name} in session {session_name}.")

    Trace(f"  Found data for subjects in {session_name}: {list(subject_folders_map.keys())}")

    for subject_id_key, folders_for_subject in subject_folders_map.items():
        Trace(f"    Processing data for subject key: {subject_id_key} in session: {session_name}")

        baseline_filepath = None
        task_condition_filepath = None

        for folder_path in folders_for_subject:
            folder_name = os.path.basename(folder_path)
            xdf_files_in_folder = glob.glob(os.path.join(folder_path, "*.xdf"))

            if not xdf_files_in_folder:
                print(f"      Warning: No XDF file found in folder {folder_name}. Skipping this folder.")
                continue
            if len(xdf_files_in_folder) > 1:
                print(f"      Warning: Multiple XDF files found in {folder_name}. Using the first one: {os.path.basename(xdf_files_in_folder[0])}.")
            current_xdf_file = xdf_files_in_folder[0]

            if "_task-Baseline" in folder_name:
                if baseline_filepath:
                    print(f"      Warning: Multiple baseline folders/files found for {subject_id_key} in {session_name}. Overwriting with data from {folder_name}.")
                baseline_filepath = current_xdf_file
                Trace(f"      Found Baseline file: {os.path.basename(baseline_filepath)} in folder {folder_name}")
            elif "_task-" in folder_name:
                if task_condition_filepath:
                    print(f"      Warning: Multiple task condition folders/files found for {subject_id_key} in {session_name}. Overwriting with data from {folder_name}.")
                task_condition_filepath = current_xdf_file
                #print(f"      Found Task Condition file: {os.path.basename(task_condition_filepath)} in folder {folder_name}")

        if baseline_filepath and task_condition_filepath:
            cleaned_subject_id = re.sub(r'^sub-', '', subject_id_key)
            Trace(f"      Pair found for {cleaned_subject_id}: Baseline ({os.path.basename(baseline_filepath)}), Task ({os.path.basename(task_condition_filepath)})")
            subject_hrv_df = process_subject_data(
                subject_id=cleaned_subject_id,
                baseline_filepath=baseline_filepath,
                task_filepath=task_condition_filepath,
                nominal_srate=nominal_srate_main
            )
            if not subject_hrv_df.empty:
                all_participant_dfs.append(subject_hrv_df)
            else:
                print(f"      No HRV data generated for subject {cleaned_subject_id} in session {session_name}.")
        else:
            missing_parts = []
            if not baseline_filepath: missing_parts.append("baseline file")
            if not task_condition_filepath: missing_parts.append("task condition file")
            print(f"      Skipping subject {subject_id_key} in session {session_name} due to missing { ' and '.join(missing_parts) }.")

if not all_participant_dfs:
    print("\nNo dataframes to concatenate. Final DataFrame will be empty.")
    final_df = pd.DataFrame()
else:
    final_df = pd.concat(all_participant_dfs, ignore_index=True)
    print("\n--- Combined Results DataFrame ---")
    if not final_df.empty:
        print(final_df.head())
        # Ensure ParticipantID is string for consistency before storing
        if 'ParticipantID' in final_df.columns:
            final_df['ParticipantID'] = final_df['ParticipantID'].astype(str)
            print("\nConverted ParticipantID in final_df to string.")
        %store final_df  # Store the DataFrame
        print("\nStored final_df.")
    else:
        print("Final DataFrame is empty after processing all subjects.")

Baseline HRV calculated for 10.
Task HRV calculated for 10.
Could not process BVP for baseline for subject 11 from _.xdf.
Could not process BVP for task for subject 11 from _.xdf.
      No HRV data generated for subject 11 in session ses-11.
Task HRV calculated for 10.
Could not process BVP for baseline for subject 11 from _.xdf.
Could not process BVP for task for subject 11 from _.xdf.
      No HRV data generated for subject 11 in session ses-11.
Could not process BVP for baseline for subject 12 from _.xdf.
Could not process BVP for task for subject 12 from _.xdf.
      No HRV data generated for subject 12 in session ses-12.
Could not process BVP for baseline for subject 13 from _.xdf.
Could not process BVP for baseline for subject 12 from _.xdf.
Could not process BVP for task for subject 12 from _.xdf.
      No HRV data generated for subject 12 in session ses-12.
Could not process BVP for baseline for subject 13 from _.xdf.
Could not process BVP for task for subject 13 from _.xdf.
  

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning:

Duplicate x values detected. Averaging their corresponding y values.



Task HRV calculated for 1.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning:

Duplicate x values detected. Averaging their corresponding y values.



Baseline HRV calculated for 2.
Task HRV calculated for 2.
Task HRV calculated for 2.


...

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning:

Duplicate x values detected. Averaging their corresponding y values.



Baseline HRV calculated for 45.


/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/neurokit2/signal/signal_interpolate.py:117: NeuroKitWarning:

Duplicate x values detected. Averaging their corresponding y values.



Task HRV calculated for 45.

--- Combined Results DataFrame ---
  ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0            10  MediumFi  Baseline  660.764801   135.314503   31.175987   
1            10  MediumFi      Task  661.646606   815.397120  163.615276   
2             1    HighFi  Baseline  715.679284   226.257083   48.481917   
3             1    HighFi      Task  727.953191  1189.775585  273.490521   
4             2    HighFi  Baseline  565.038654    57.817954   16.013175   

   HRV_SDNNI1  HRV_SDANN2  HRV_SDNNI2  HRV_SDANN5  ...  HRV_SampEn  \
0  123.251099   17.586581  124.454787         NaN  ...    0.748622   
1  525.514743   87.186293  544.865326         NaN  ...    0.260511   
2  220.196165   40.496413  220.077430         NaN  ...    0.637484   
3  930.099612  110.209036  989.813782         NaN  ...    0.284032   
4   49.148874   19.703470   52.108503         NaN  ...    0.697417   

   HRV_ShanEn  HRV_FuzzyEn  HRV_MSEn  HRV_CMSEn  HRV_RCMSE

UsageError: Unknown variable '#'


In [ ]:
# Check a single subject's bvp stream to see if time stamps are evenly spaced
# Used for debugging
def check_bvp_timestamps(filepath):
    bvp_signal, ts, _, streams_found = extract_bvp_and_markers_from_xdf(filepath)
    if streams_found['bvp'] and bvp_signal is not None and ts is not None:
        print(f"BVP stream found in {filepath}.")
        time_diffs = np.diff(ts)
        print(f"Time differences between BVP samples: {time_diffs}")
        print(f"Mean time difference: {np.mean(time_diffs)}")
        print(f"Standard deviation of time differences: {np.std(time_diffs)}")
    else:
        print(f"No BVP stream found in {filepath}.")
check_bvp_timestamps("/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-27/sub-27_ses-27_task-MediumFi/_.xdf")

BVP stream found in /home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/Test_Data/ses-27/sub-27_ses-27_task-MediumFi/_.xdf.
Time differences between BVP samples: [0.0024997  0.0024997  0.0024997  ... 0.00249975 0.00249975 0.00249975]
Mean time difference: 0.002533121224245887
Standard deviation of time differences: 0.021291432581365042


## Statistical Model

With the DataFrame (`final_df`) containing HRV metrics for each participant, condition, and phase, we can compare the baseline vs task across the conditions.

A Linear Mixed-Effects Model (LMM) is used here. It allows us to model:
- **Fixed Effects:** The average effects of Phase (Baseline vs. Task) and Condition (LowFi, MedFi, HighFi), and their interaction (Phase * Condition).
- **Random Effects:** The variability between participants. They each have their own baseline level of the HRV metric unique to them, which is modeled as a random intercept (~1 ParticipantID).

In [ ]:
import statsmodels.formula.api as smf

# Check if final_df exists and has data
if 'final_df' in locals() and not final_df.empty and 'HRV_RMSSD' in final_df.columns:
    # Ensure necessary columns are not all NaN
    if final_df[['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID']].isnull().all().any():
        print("Warning: One or more critical columns contain only NaN values. Cannot run model.")
    else: 
        # Remove rows with NaN in the outcome variable or predictors
        model_df = final_df.dropna(subset=['HRV_RMSSD', 'Phase', 'Condition', 'ParticipantID'])
        
        if model_df.empty:
            print("Warning: No valid data remaining after removing NaNs. Cannot run model.")
        else:
            #print("\n--- Inspecting model_df before fitting LMM ---")
            #print(model_df.info())

            # Print a few rows to see actual values
            print("\n--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---")
            # Ensure ParticipantID, Phase, and Condition are appropriate types
            model_df['ParticipantID'] = model_df['ParticipantID'].astype('category')
            model_df['Phase'] = model_df['Phase'].astype('category')
            model_df['Condition'] = model_df['Condition'].astype('category')
            
            # Define the model formula for fixed effects
            fixed_effects_formula = "HRV_RMSSD ~ C(Phase) * C(Condition)"
            # Define the random effects formula (random intercept for ParticipantID)
            random_effects_formula = "~1"
            
            try:
                # Fit the model using re_formula for random effects
                model = smf.mixedlm(fixed_effects_formula, 
                                  model_df, 
                                  groups=model_df["ParticipantID"], 
                                  re_formula=random_effects_formula)
                result = model.fit()
                
                # Print the summary
                print(result.summary())
            except Exception as e:
                print(f"Error fitting model: {e}")
                print("\nPlease check data structure and variability.")
                print("Model DataFrame head:")
                print(model_df.head())
else:
    print("Skipping statistical analysis: 'final_df' not created or is empty or missing 'HRV_RMSSD' column.")


--- Fitting Linear Mixed-Effects Model for HRV_RMSSD ---
                            Mixed Linear Model Regression Results
Model:                         MixedLM            Dependent Variable:            HRV_RMSSD   
No. Observations:              75                 Method:                        REML        
No. Groups:                    39                 Scale:                         1591039.6271
Min. group size:               1                  Log-Likelihood:                -601.9519   
Max. group size:               2                  Converged:                     Yes         
Mean group size:               1.9                                                           
---------------------------------------------------------------------------------------------
                                            Coef.    Std.Err.   z    P>|z|   [0.025   0.975] 
---------------------------------------------------------------------------------------------
Intercept                     

# Results

This mini-project aimed to create a pipeline for processing BVP data to extract HRV metrics; furthermore exploring the potential correlations with the user experience in a VR experience. The data was extracted using a Butterworth bandpass filter and utilized HeartPy and NeuoroKit2 to calculate the HRV RMSSD.

A LMM was used to analyze the data, considering the fixed effects of phase as baseline vs task, conditions as LowFi, MedFi, HighFi, and their interaction. While still accounting for random variability between participants.

The model showed indications that baseline RMSSD levels were comparable across the different conditions, which was expected. While not statistically significant at the p < 0.05 threshold, the interaction `C(Phase)[T.Task]:C(condition)[T.LowFi]` (HighFi reference group) saw a trend level of p = 0.074, where the LowFi condition might evoke a different physiological response. Specifically, the results hinted that the LowFi group experienced a smaller change in RMSSD during their task compared to the HighFi group.

## Survey Data Analysis: System Immersion and Performance Confidence

### 1. Setup: Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import pingouin as pg
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Load the post-test survey data
post_test_file = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/MED 802 Post Test survey (Part 2)(1-45).csv"
try:
    df_post_test = pd.read_csv(post_test_file)
    print("Post-test survey data loaded successfully.")
    print(f"Shape of the dataframe: {df_post_test.shape}")
    # Display the first few rows and column names to understand its structure
    print("\\nFirst 5 rows of the post-test data:")
    print(df_post_test.head())
    print("\\nColumn names:")
    print(df_post_test.columns.tolist())
except FileNotFoundError:
    print(f"Error: The file {post_test_file} was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the post-test survey data: {e}")

# Potential: Load pre-test survey data if available and needed for H1a
pre_test_file = "/home/mitchell/Documents/Projects/P8-Project/Dataprocessing Pipeline/MED 802 Pre Test survey (Part 1)(1-45).csv"
df_pre_test = None # Initialize as None
try:
    df_pre_test = pd.read_csv(pre_test_file)
    print("\\nPre-test survey data loaded successfully.")
    print(f"Shape of the pre-test dataframe: {df_pre_test.shape}")
    # Display the first few rows and column names
    print("\\nFirst 5 rows of the pre-test data:")
    print(df_pre_test.head())
    print("\\nColumn names (pre-test):")
    print(df_pre_test.columns.tolist())
except FileNotFoundError:
    print(f"Info: The pre-test file {pre_test_file} was not found. Proceeding without it for now.")
except Exception as e:
    print(f"An error occurred while loading the pre-test survey data: {e}")


Post-test survey data loaded successfully.
Shape of the dataframe: (45, 46)
\nFirst 5 rows of the post-test data:
   ID       Start time  Completion time      Email  Name  Last modified time  \
0   6  5-5-25 10:03:43  5-5-25 11:54:50  anonymous   NaN                 NaN   
1   7  5-5-25 13:17:55  5-5-25 13:35:54  anonymous   NaN                 NaN   
2   8  5-5-25 14:01:48  5-5-25 14:25:06  anonymous   NaN                 NaN   
3   9  5-5-25 14:35:10  5-5-25 14:55:15  anonymous   NaN                 NaN   
4  10  5-5-25 15:16:02  5-5-25 15:19:07  anonymous   NaN                 NaN   

   Participant Number (Test conductor fills out)  \
0                                              1   
1                                              2   
2                                              3   
3                                              4   
4                                              5   

  What group is this? (Test conductor fills out)\n  \
0                                     

### 2. Data Cleaning and Preprocessing

In [26]:
# Data Cleaning and Preprocessing
print("Starting Data Cleaning and Preprocessing...")

# Define a more comprehensive column mapping
# Based on the output of df_post_test.columns and df_pre_test.columns from the previous cell
column_mapping = {
    # Common columns
    # For df_post_test, 'ID' is the survey record ID, 'ParticipantID' is the manually entered one.
    # For df_pre_test, 'Record ID' is the survey record ID, 'Participant ID' is the manually entered one.
    'Record ID': 'SurveyRecordID', # Renaming to avoid clash if both dfs had 'ID'
    'Participant ID': 'ParticipantID', # This is the key for merging

    # Post-test survey specific columns
    'What group is this? (Test conductor fills out)': 'SystemImmersionGroup',
    'How confident are you that you performed well in the game? (7 = Very confident, 1 = Not at all confident)': 'PerformanceConfidence',
    'How immersed did you feel?': 'PerceivedImmersion',
    'To what extent did the game hold your attention? ( 7= A lot, 1 = Not at all )': 'AttentionHold',
    'To what extent did you feel you were focused on the game? ( 7= A lot, 1 = Not at all )': 'FocusOnGame',
    'How much effort did you put into playing the game? ( 7= A lot, 1 = Very little )': 'EffortInGame',
    'Did you feel that you were trying your best? ( 7= Very much so, 1 =Not at all )': 'TryingBest',
    'To what extent did you lose track of time, e.g. did the game absorb your attention so that you were not bored? ( 7= A lot, 1 = Not at all)': 'TimeLoss',
    'To what extent did you feel consciously aware of being in the real world whilst playing? ( 7= Very much so, 1 = Not at all)': 'RealWorldAwareness',
    'To what extent did you forget about your everyday concerns? ( 7= A lot, 1 = Not at all)': 'ForgotConcerns',
    'To what extent were you aware of yourself in your surroundings?  ( 7= Very Aware, 1 = Not at all)': 'SelfAwarenessInSurroundings',
    'To what extent did you notice events taking place around you?   ( 7= A lot, 1 = Not at all)': 'NoticedSurroundingEvents',
    'Did you feel the urge at any point to stop playing and see what was happening around you?    ( 7= Very much so, 1 = Not at all)': 'UrgeToStop',
    'To what extent did you feel that you were interacting with the game environment?    ( 7= Very much so, 1 = Not at all)': 'InteractionWithGame',
    'To what extent did you feel as though you were separated from your real-world environment?    ( 7= Very much so, 1 = Not at all)': 'SeparationFromRealWorld',
    'To what extent did you feel that the game was something fun you were experiencing, rather than a task you were just doing?    ( 7= Very much so, 1 = Not at all)': 'GameAsFun',
    'To what extent was your sense of being in the game environment stronger than your sense of being in the real world?    ( 7= Very much so, 1 = Not at all)': 'SenseOfGameStronger',
    'At any point did you find yourself become so involved that you were unaware you were even using controls, e.g. it was effortless?    ( 7= Very much so, 1 = Not at all)': 'UnawareOfControls',
    'To what extent did you feel as though you were moving through the game according to your own will? ( 7= Very much so, 1 = Not at all)': 'MovementByWill',
    'To what extent did you find the game challenging? ( 7= Very difficult, 1 = Not at all)': 'GameChallenge',
    'Were there any times during the game in which you just wanted to give up? ( 7= A lot, 1 = Not at all)': 'WantedToGiveUp',
    'To what extent did you feel motivated while playing? ( 7= A lot, 1 = Not at all)': 'MotivationInGame',
    'To what extent did you find the game easy?\xa0 ( 7= Very much so, 1 = Not at all)': 'GameEasy',
    'To what extent did you feel like you were making progress towards the end of the game?\xa0 ( 7= A lot, 1 = Not at all)': 'ProgressTowardsEnd',
    'To what extent did you feel emotionally attached to the game? ( 7= Very much so, 1 = Not at all)': 'EmotionalAttachment',
    'To what extent were you interested in seeing how the game’s events would progress? ( 7= A lot, 1 = Not at all)': 'InterestInGameEvents',
    'How much did you want to “win” the game? ( 7= Very much so, 1 = Not at all)': 'DesireToWin',
    'Were you in suspense about whether or not you would do well in the game? ( 7= Very much so, 1 = Not at all)': 'SuspenseAboutPerformance',
    'At any point did you find yourself become so involved that you wanted to speak to the game directly? ( 7= Very much so, 1 = Not at all)': 'WantedToSpeakToGame',
    'To what extent did you enjoy the graphics and the imagery? ( 7= A lot, 1 = Not at all)': 'EnjoyedGraphics',
    'How much would you say you enjoyed playing the game? ( 7= A lot, 1 = Not at all)': 'EnjoyedGameOverall',
    'When it ended, were you disappointed that the game was over? ( 7= Very much so, 1 = Not at all)': 'DisappointedGameEnded',
    'Would you like to play the game again? ( 7 = Definitely yes, 1 = Definitely no)': 'PlayAgain',
    # NASA-TLX Columns
    'How mentally demanding was the task? (1 being "Very low" and 20 being "Very high")': 'MentalDemand',
    'How physically demanding was the task? (1 being "Very low" and 20 being "Very high")': 'PhysicalDemand',
    'How hurried or rushed was the pace of the task? (1 being "Very low" and 20 being "Very high")': 'TemporalDemand',
    'How successful were you in accomplishing what you were asked to do? (1 being "Very low" and 20 being "Very high")': 'PerformanceNASA',
    'How hard did you have to work to accomplish your level of performance? (1 being "Very low" and 20 being "Very high")': 'EffortNASA',
    'How insecure, discouraged, irritated, stressed, and annoyed were you? (1 being "Very low" and 20 being "Very high")': 'FrustrationNASA',

    # Pre-test survey specific columns (Add mappings if pre-test data is to be used for UserSkill)
    # Example: 'What is your current skill level in playing first-person shooter (FPS) games on a computer (mouse and keyboard)?': 'UserSkill_PreTest'
    # Add the actual column name from your pre-test CSV that represents user skill.
    # For now, we will assume a generic pre-test question might be used for UserSkill.
    # If you have a specific column, replace 'PreTest_UserSkill_Question' with its exact name from df_pre_test.columns
    'PreTest_UserSkill_Question': 'UserSkill_PreTest' # Placeholder - replace if a real skill question exists
}

# Apply renaming to a copy to avoid SettingWithCopyWarning later if we filter
df_post_test_cleaned = df_post_test.copy()
df_post_test_cleaned.rename(columns=column_mapping, inplace=True)
# Rename the original 'ID' column from post-test to avoid confusion with 'ParticipantID'
if 'ID' in df_post_test_cleaned.columns and 'ParticipantID' in df_post_test_cleaned.columns:
    df_post_test_cleaned.rename(columns={'ID': 'PostTestSurveyRecordID'}, inplace=True)

print("\nPost-test DataFrame columns after renaming attempt:")
print(list(df_post_test_cleaned.columns))

# Convert relevant columns to numeric, coercing errors
columns_to_numeric = ['PerformanceConfidence', 'PerceivedImmersion'] 
likert_scale_columns = [
    'AttentionHold', 'FocusOnGame', 'EffortInGame', 'TryingBest', 'TimeLoss',
    'RealWorldAwareness', 'ForgotConcerns', 'SelfAwarenessInSurroundings',
    'NoticedSurroundingEvents', 'UrgeToStop', 'InteractionWithGame',
    'SeparationFromRealWorld', 'GameAsFun', 'SenseOfGameStronger',
    'UnawareOfControls', 'MovementByWill', 'GameChallenge', 'WantedToGiveUp',
    'MotivationInGame', 'GameEasy', 'ProgressTowardsEnd', 'EmotionalAttachment',
    'InterestInGameEvents', 'DesireToWin', 'SuspenseAboutPerformance',
    'WantedToSpeakToGame', 'EnjoyedGraphics', 'EnjoyedGameOverall',
    'DisappointedGameEnded', 'PlayAgain'
]
nasa_tlx_columns = ['MentalDemand', 'PhysicalDemand', 'TemporalDemand', 'PerformanceNASA', 'EffortNASA', 'FrustrationNASA']

# ParticipantID should also be numeric
columns_to_convert = ['ParticipantID'] + columns_to_numeric + likert_scale_columns + nasa_tlx_columns

converted_cols_log = []
for col in columns_to_convert:
    if col in df_post_test_cleaned.columns:
        df_post_test_cleaned[col] = pd.to_numeric(df_post_test_cleaned[col], errors='coerce')
        converted_cols_log.append(col)
    else:
        print(f"Warning: Column '{col}' not found in df_post_test_cleaned for numeric conversion.")
print(f"\nAttempted to convert to numeric: {converted_cols_log}")

# Handle missing data
crucial_columns = ['ParticipantID', 'PerformanceConfidence', 'SystemImmersionGroup', 'PerceivedImmersion']
print(f"\nChecking for missing values in crucial columns: {crucial_columns}")
print(df_post_test_cleaned[crucial_columns].isnull().sum())

initial_rows = len(df_post_test_cleaned)
df_post_test_cleaned.dropna(subset=crucial_columns, inplace=True)
print(f"Dropped {initial_rows - len(df_post_test_cleaned)} rows from df_post_test_cleaned due to missing data in crucial columns.")
print(f"Shape of df_post_test_cleaned after dropna: {df_post_test_cleaned.shape}")

# Clean pre_test data similarly
df_pre_test_cleaned = df_pre_test.copy()
df_pre_test_cleaned.rename(columns=column_mapping, inplace=True) # Use the same mapping, it will only affect relevant columns

# Ensure ParticipantID is present and numeric in pre-test for merging
if 'ParticipantID' in df_pre_test_cleaned.columns:
    df_pre_test_cleaned['ParticipantID'] = pd.to_numeric(df_pre_test_cleaned['ParticipantID'], errors='coerce')
    df_pre_test_cleaned.dropna(subset=['ParticipantID'], inplace=True)
    print(f"\nPre-test data ParticipantID type: {df_pre_test_cleaned['ParticipantID'].dtype}")

    # Check if 'UserSkill_PreTest' column exists after renaming (it might not if the placeholder wasn't matched)
    if 'UserSkill_PreTest' in df_pre_test_cleaned.columns:
        df_pre_test_cleaned['UserSkill_PreTest'] = pd.to_numeric(df_pre_test_cleaned['UserSkill_PreTest'], errors='coerce')
        # Merge with post-test data
        # Ensure ParticipantID in df_post_test_cleaned is also of a compatible type for merging (it should be int/float after numeric conversion)
        print(f"Post-test data ParticipantID type before merge: {df_post_test_cleaned['ParticipantID'].dtype}")
        
        df_merged = pd.merge(df_post_test_cleaned, df_pre_test_cleaned[['ParticipantID', 'UserSkill_PreTest']], on='ParticipantID', how='left')
        if 'UserSkill_PreTest' in df_merged.columns and not df_merged['UserSkill_PreTest'].isnull().all():
            print(f"Successfully merged pre-test UserSkill data. Shape of merged_df: {df_merged.shape}")
            # Check how many UserSkill_PreTest values are not NaN after merge
            print(f"Number of non-NaN UserSkill_PreTest values after merge: {df_merged['UserSkill_PreTest'].notna().sum()}")
            df_post_test_cleaned = df_merged # Update df_post_test_cleaned with the merged data
        else:
            print("Warning: Merge attempted, but 'UserSkill_PreTest' column is missing in merged_df or all its values are NaN. H1a might not be testable as intended.")
    else:
        print("Warning: 'UserSkill_PreTest' column not found in pre-test data after renaming. Cannot merge for H1a.")
else:
    print("Warning: 'ParticipantID' column not found in pre-test data after renaming/cleaning. Cannot merge for H1a.")

print("\nFinal data types for df_post_test_cleaned (potentially merged):")
print(df_post_test_cleaned.dtypes)
print("\nCleaned and potentially merged post-test data head (df_post_test_cleaned):")
print(df_post_test_cleaned.head())

# Store cleaned data for subsequent cells
%store df_post_test_cleaned
if 'df_pre_test_cleaned' in locals() and 'UserSkill_PreTest' in df_pre_test_cleaned.columns:
    %store df_pre_test_cleaned

print("\nData Cleaning and Preprocessing Complete.")


Starting Data Cleaning and Preprocessing...

Post-test DataFrame columns after renaming attempt:
['PostTestSurveyRecordID', 'Start time', 'Completion time', 'Email', 'Name', 'Last modified time', 'ParticipantID', 'SystemImmersionGroup', 'AttentionHold', 'FocusOnGame', 'EffortInGame', 'TryingBest', 'TimeLoss', 'RealWorldAwareness', 'ForgotConcerns', 'SelfAwarenessInSurroundings', 'NoticedSurroundingEvents', 'UrgeToStop', 'InteractionWithGame', 'SeparationFromRealWorld', 'GameAsFun', 'SenseOfGameStronger', 'UnawareOfControls', 'MovementByWill', 'GameChallenge', 'WantedToGiveUp', 'MotivationInGame', 'GameEasy', 'ProgressTowardsEnd', 'PerformanceConfidence', 'EmotionalAttachment', 'InterestInGameEvents', 'DesireToWin', 'SuspenseAboutPerformance', 'WantedToSpeakToGame', 'EnjoyedGraphics', 'EnjoyedGameOverall', 'DisappointedGameEnded', 'PlayAgain', 'PerceivedImmersion', 'MentalDemand', 'PhysicalDemand', 'TemporalDemand', 'PerformanceNASA', 'EffortNASA', 'FrustrationNASA']

Attempted to conve

### 3. Define Variables

In [34]:
# Define Variables

# Retrieve the cleaned DataFrame stored by the previous cell
%store -r df_post_test_cleaned

# Ensure df_post_test_cleaned is available
if 'df_post_test_cleaned' not in locals():
    print("Error: df_post_test_cleaned not found. Please re-run the data cleaning cell.")
else:
    # Define ImmersionLevel based on SystemImmersionGroup
    # Group A: Low Immersion (Desktop)
    # Group B: Medium Immersion (Single Screen VR)
    # Group C: High Immersion (CAVE)
    if 'SystemImmersionGroup' in df_post_test_cleaned.columns:
        conditions = [
            df_post_test_cleaned['SystemImmersionGroup'] == 'Group A',
            df_post_test_cleaned['SystemImmersionGroup'] == 'Group B',
            df_post_test_cleaned['SystemImmersionGroup'] == 'Group C'
        ]
        choices = ['Low', 'Medium', 'High']
        df_post_test_cleaned['ImmersionLevel'] = np.select(conditions, choices, default='Unknown')
        # Convert ImmersionLevel to a categorical type with a specific order for plotting and analysis
        immersion_order = ['Low', 'Medium', 'High']
        df_post_test_cleaned['ImmersionLevel'] = pd.Categorical(df_post_test_cleaned['ImmersionLevel'], categories=immersion_order, ordered=True)
        print("'ImmersionLevel' column created and ordered.")
        print(df_post_test_cleaned[['SystemImmersionGroup', 'ImmersionLevel']].head())
    else:
        print("Warning: 'SystemImmersionGroup' column not found in df_post_test_cleaned. Cannot create 'ImmersionLevel'.")

    # Convert ParticipantID to string to ensure it's treated as a categorical factor in LMM
    if 'ParticipantID' in df_post_test_cleaned.columns:
        df_post_test_cleaned['ParticipantID'] = df_post_test_cleaned['ParticipantID'].astype(str)
        print("\nConverted 'ParticipantID' to string type.")
        print(df_post_test_cleaned['ParticipantID'].dtype)
    else:
        print("Warning: 'ParticipantID' column not found in df_post_test_cleaned.")

    # Categorize PerceivedImmersion (example: Low, Medium, High based on quantiles or fixed breaks)
    # This is for H4. The scale is 1-7 (or 1-10 based on CSV output, needs clarification from data dictionary or survey)
    # Assuming 1-10 scale from previous output for 'How immersed did you feel?'
    if 'PerceivedImmersion' in df_post_test_cleaned.columns:
        # Check min/max to confirm scale
        min_pi = df_post_test_cleaned['PerceivedImmersion'].min()
        max_pi = df_post_test_cleaned['PerceivedImmersion'].max()
        print(f"\nPerceivedImmersion scale: Min={min_pi}, Max={max_pi}")
        
        # Define bins and labels based on the actual scale observed
        # Example for a 1-10 scale. Adjust if it's 1-7.
        if pd.api.types.is_numeric_dtype(df_post_test_cleaned['PerceivedImmersion']):
            bins = [0, 3, 7, 10] # Adjust these bins as appropriate for your scale (0-3 Low, 4-7 Med, 8-10 High)
            labels = ['LowPI', 'MediumPI', 'HighPI']
            if max_pi <= 7: # Adjust for a 1-7 scale if necessary
                 bins = [0, 2, 5, 7]
                 labels = ['LowPI', 'MediumPI', 'HighPI']
            
            df_post_test_cleaned['PerceivedImmersionCategory'] = pd.cut(df_post_test_cleaned['PerceivedImmersion'], bins=bins, labels=labels, right=True)
            print("'PerceivedImmersionCategory' column created.")
            print(df_post_test_cleaned[['PerceivedImmersion', 'PerceivedImmersionCategory']].head())
        else:
            print("Warning: 'PerceivedImmersion' is not numeric, cannot categorize.")
    else:
        print("Warning: 'PerceivedImmersion' column not found. Cannot create 'PerceivedImmersionCategory'.")

    # Store the updated DataFrame for the next cells
    %store df_post_test_cleaned

    print("\nDataFrame info after variable definition:")
    df_post_test_cleaned.info()
    print("\nDataFrame head after variable definition:")
    print(df_post_test_cleaned.head())

'ImmersionLevel' column created and ordered.
  SystemImmersionGroup ImmersionLevel
0              Group C           High
1              Group C           High
2              Group B         Medium
3              Group C           High
4              Group C           High

Converted 'ParticipantID' to string type.
object

PerceivedImmersion scale: Min=5, Max=10
'PerceivedImmersionCategory' column created.
   PerceivedImmersion PerceivedImmersionCategory
0                   7                   MediumPI
1                   6                   MediumPI
2                  10                     HighPI
3                   7                   MediumPI
4                   9                     HighPI
Stored 'df_post_test_cleaned' (DataFrame)

DataFrame info after variable definition:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 48 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  

### 4. Exploratory Data Analysis (EDA)

In [31]:
if 'df_post_test_cleaned' in locals() and not df_post_test_cleaned.empty and 'ImmersionLevel' in df_post_test_cleaned.columns:
    # Columns for EDA box plots
    eda_cols = [
        'PerformanceConfidence', 'PerceivedImmersion', 'MentalDemand', 
        'PhysicalDemand', 'TemporalDemand', 'PerceivedPerformanceRaw', 
        'EffortNASA', 'FrustrationNASA'
    ]
    
    # Filter to existing columns to avoid errors
    actual_eda_cols = [col for col in eda_cols if col in df_post_test_cleaned.columns]

    if not actual_eda_cols:
        print("No columns available for EDA plots after checking existence in DataFrame.")
    else:
        print(f"\\nGenerating EDA box plots for: {actual_eda_cols}")
        for col in actual_eda_cols:
            if df_post_test_cleaned[col].isnull().all():
                print(f"Skipping box plot for {col} as all values are NaN.")
                continue
            try:
                fig = px.box(df_post_test_cleaned, x='ImmersionLevel', y=col, 
                             title=f'{col} by Immersion Level', 
                             points="all",  # Show all data points
                             labels={'ImmersionLevel': 'Immersion Level'})
                fig.show()
            except Exception as e:
                print(f"Could not generate box plot for {col}: {e}")
else:
    print("Skipping EDA as df_post_test_cleaned is not available, empty, or 'ImmersionLevel' column is missing.")

Skipping EDA as df_post_test_cleaned is not available, empty, or 'ImmersionLevel' column is missing.


### 5. Hypothesis Testing (Linear Mixed Models - LMMs)

We will use Linear Mixed Models (LMMs) to test our hypotheses. The basic structure for most models will be `DependentVariable ~ ImmersionLevel`, with `ParticipantID` as a random intercept to account for individual differences.

In [35]:
# Define a helper function to fit and summarize LMMs
def fit_lmm(formula, data, group_col='ParticipantID', dependent_var_name='DependentVar'):
    """Fits an LMM and prints summary, checks for issues."""
    if not all(c in data.columns for c in [dependent_var_name, 'ImmersionLevel', group_col]):
        print(f"Error: One or more required columns ('{dependent_var_name}', 'ImmersionLevel', '{group_col}') not in DataFrame for formula: {formula}")
        return None
    
    # Drop rows where the current dependent variable or ImmersionLevel or ParticipantID is NaN
    # This is crucial for each specific model
    model_data = data.dropna(subset=[dependent_var_name, 'ImmersionLevel', group_col])

    if model_data.empty:
        print(f"No data remaining for {dependent_var_name} after dropping NaNs. Skipping LMM.")
        return None
    if model_data[group_col].nunique() < 2:
        print(f"Not enough groups (participants) for {dependent_var_name} after dropping NaNs. Need at least 2. Skipping LMM.")
        return None
    if model_data['ImmersionLevel'].nunique() < 2:
        print(f"Not enough levels in 'ImmersionLevel' for {dependent_var_name} after dropping NaNs. Need at least 2. Skipping LMM.")
        return None
    # Check if the dependent variable has variance
    if model_data[dependent_var_name].var() == 0:
        print(f"Warning: Dependent variable '{dependent_var_name}' has zero variance. LMM may not converge or be meaningful.")

    print(f"\\n--- LMM for: {dependent_var_name} --- Formula: {formula}")
    try:
        model = smf.mixedlm(formula, model_data, groups=model_data[group_col])
        result = model.fit(method=["bfgs"]) # Try BFGS, then L-BFGS-B if it fails
        print(result.summary())
        return result
    except Exception as e:
        print(f"Error fitting LMM for {dependent_var_name}: {e}")
        try:
            print("Retrying with L-BFGS-B...")
            result = model.fit(method=["lbfgs"])
            print(result.summary())
            return result
        except Exception as e2:
            print(f"Error fitting LMM for {dependent_var_name} with L-BFGS-B: {e2}")
            print("Model data sample:")
            print(model_data.head())
            print("Value counts for ImmersionLevel:")
            print(model_data['ImmersionLevel'].value_counts())
            print("Value counts for ParticipantID (first 5):")
            print(model_data[group_col].value_counts().head())
            return None

# Dictionary to store model results
lmm_results = {}

%store -r df_post_test_cleaned
#%store -r final_df # Load the HRV data

# Ensure df_post_test_cleaned is available
if 'df_post_test_cleaned' not in locals():
    print("Error: df_post_test_cleaned not found. Please re-run the data cleaning and variable definition cells.")
elif 'final_df' not in locals() or final_df.empty:
    print("Error: final_df (HRV data) not found or is empty. Please ensure the HRV processing cells were run and final_df was stored.")
else:
    print("Successfully loaded df_post_test_cleaned and final_df (HRV data).")
    print(f"Shape of df_post_test_cleaned: {df_post_test_cleaned.shape}")
    print(f"Shape of final_df (HRV): {final_df.shape}")
    print("\nHead of final_df (HRV data):")
    print(final_df.head())
    print("\nData types of final_df (HRV data):")
    print(final_df.dtypes)
    
    # --- Merge HRV data (Task phase only) with survey data ---
    # We are interested in HRV during the task as a covariate for survey responses collected post-task.
    hrv_task_data = final_df[final_df['Phase'] == 'Task']
    
    # Ensure ParticipantID in hrv_task_data is string for merging, if not already
    if 'ParticipantID' in hrv_task_data.columns and hrv_task_data['ParticipantID'].dtype != 'object':
        hrv_task_data = hrv_task_data.copy() # Avoid SettingWithCopyWarning
        hrv_task_data.loc[:, 'ParticipantID'] = hrv_task_data['ParticipantID'].astype(str)
        print("\nConverted ParticipantID in hrv_task_data to string for merging.")

    # Ensure ParticipantID in df_post_test_cleaned is also string (should be from Variable Definition cell)
    if 'ParticipantID' in df_post_test_cleaned.columns and df_post_test_cleaned['ParticipantID'].dtype != 'object':
        df_post_test_cleaned = df_post_test_cleaned.copy()
        df_post_test_cleaned.loc[:, 'ParticipantID'] = df_post_test_cleaned['ParticipantID'].astype(str)
        print("\nConverted ParticipantID in df_post_test_cleaned to string for merging.")

    # Select only relevant HRV columns to merge (e.g., HRV_RMSSD)
    hrv_cols_to_merge = ['ParticipantID', 'HRV_RMSSD'] # Add other HRV metrics if needed
    hrv_task_subset = hrv_task_data[hrv_cols_to_merge]

    # Perform the merge
    df_analysis = pd.merge(df_post_test_cleaned, hrv_task_subset, on='ParticipantID', how='left')
    print(f"\nShape of df_analysis after merging with HRV task data: {df_analysis.shape}")
    print("Number of rows with missing HRV_RMSSD after merge:", df_analysis['HRV_RMSSD'].isnull().sum())
    
    # Handle potential missing HRV data after merge (e.g., by imputation or by noting it)
    if 'HRV_RMSSD' in df_analysis.columns and df_analysis['HRV_RMSSD'].isnull().any():
        median_hrv_rmssd = df_analysis['HRV_RMSSD'].median()
        df_analysis['HRV_RMSSD'].fillna(median_hrv_rmssd, inplace=True)
        print(f"Filled {df_analysis['HRV_RMSSD'].isnull().sum()} missing HRV_RMSSD values with median ({median_hrv_rmssd:.2f}).")

    # Define a helper function to fit and summarize LMMs
    def fit_lmm(formula, data, group_col='ParticipantID', dependent_var_name='DependentVar'):
        """Fits an LMM and prints summary, checks for issues."""
        # Check for all columns in the formula string + group_col
        # This is a bit simplistic; a proper formula parser would be better for complex formulas.
        required_cols_from_formula = [col.strip() for col in formula.replace("~", " ").replace("*", " ").replace("+", " ").split()
                                      if col.strip() not in ['1', dependent_var_name, group_col] and not col.startswith("C(")]
        required_cols = list(set([dependent_var_name, 'ImmersionLevel', group_col] + required_cols_from_formula))
        
        missing_cols = [col for col in required_cols if col not in data.columns]
        if missing_cols:
            print(f"Error: One or more required columns ({missing_cols}) not in DataFrame for formula: {formula}")
            return None
        
        model_data = data.dropna(subset=required_cols)

        if model_data.empty:
            print(f"No data remaining for {dependent_var_name} after dropping NaNs from {required_cols}. Skipping LMM.")
            return None
        if model_data[group_col].nunique() < 2:
            print(f"Not enough groups (participants) for {dependent_var_name} after dropping NaNs. Need at least 2. Skipping LMM.")
            return None
        if 'ImmersionLevel' in model_data.columns and model_data['ImmersionLevel'].nunique() < 2:
            print(f"Not enough levels in 'ImmersionLevel' for {dependent_var_name} after dropping NaNs. Need at least 2. Skipping LMM.")
            #return None # Allow running if ImmersionLevel is not in formula
        if model_data[dependent_var_name].var() == 0:
            print(f"Warning: Dependent variable '{dependent_var_name}' has zero variance. LMM may not converge or be meaningful.")

        print(f"\n--- LMM for: {dependent_var_name} --- Formula: {formula}")
        try:
            model = smf.mixedlm(formula, model_data, groups=model_data[group_col])
            result = model.fit(method=["bfgs"]) 
            print(result.summary())
            return result
        except Exception as e:
            print(f"Error fitting LMM for {dependent_var_name} with BFGS: {e}")
            try:
                print("Retrying with L-BFGS-B...")
                result = model.fit(method=["lbfgs"])
                print(result.summary())
                return result
            except Exception as e2:
                print(f"Error fitting LMM for {dependent_var_name} with L-BFGS-B: {e2}")
                print("Model data sample:")
                print(model_data.head())
                if 'ImmersionLevel' in model_data.columns: print("Value counts for ImmersionLevel:", model_data['ImmersionLevel'].value_counts())
                print("Value counts for ParticipantID (first 5):", model_data[group_col].value_counts().head())
                return None

    # Dictionary to store model results
    lmm_results = {}

    # Use df_analysis for all models now
    if not df_analysis.empty:
        # H1 & H1a: Performance Confidence, now with HRV_RMSSD as covariate
        if 'PerformanceConfidence' in df_analysis.columns and 'HRV_RMSSD' in df_analysis.columns:
            if 'UserSkill_PreTest' in df_analysis.columns and not df_analysis['UserSkill_PreTest'].isnull().all():
                print("\nUserSkill_PreTest column found, using it for H1a with HRV_RMSSD.")
                df_analysis['UserSkill_PreTest_Numeric'] = pd.to_numeric(df_analysis['UserSkill_PreTest'], errors='coerce')
                median_skill = df_analysis['UserSkill_PreTest_Numeric'].median()
                df_analysis['UserSkill_PreTest_Numeric'].fillna(median_skill, inplace=True)
                
                formula_h1a_cov = "PerformanceConfidence ~ ImmersionLevel * UserSkill_PreTest_Numeric + HRV_RMSSD"
                lmm_results['H1a_PerfConf_Skill_HRV_Interaction'] = fit_lmm(formula_h1a_cov, df_analysis, dependent_var_name='PerformanceConfidence')
                
                formula_h1a_no_int_cov = "PerformanceConfidence ~ ImmersionLevel + UserSkill_PreTest_Numeric + HRV_RMSSD"
                lmm_results['H1a_PerfConf_Skill_HRV_NoInteraction'] = fit_lmm(formula_h1a_no_int_cov, df_analysis, dependent_var_name='PerformanceConfidence')
            else:
                print("\nUserSkill_PreTest column not found or all NaN. Proceeding with H1 (PerformanceConfidence ~ ImmersionLevel + HRV_RMSSD).")
                formula_h1_cov = "PerformanceConfidence ~ ImmersionLevel + HRV_RMSSD"
                lmm_results['H1_PerformanceConfidence_HRV'] = fit_lmm(formula_h1_cov, df_analysis, dependent_var_name='PerformanceConfidence')
        else:
            print("Skipping H1/H1a: 'PerformanceConfidence' or 'HRV_RMSSD' column not found in df_analysis.")

        # H2: Task Performance (using PerformanceNASA as proxy, with HRV_RMSSD)
        # Renamed 'PerceivedPerformanceRaw' to 'PerformanceNASA' in cleaning, assuming this is the intended NASA-TLX performance item.
        if 'PerformanceNASA' in df_analysis.columns and 'HRV_RMSSD' in df_analysis.columns:
            formula_h2_cov = "PerformanceNASA ~ ImmersionLevel + HRV_RMSSD"
            lmm_results['H2_PerformanceNASA_HRV'] = fit_lmm(formula_h2_cov, df_analysis, dependent_var_name='PerformanceNASA')
            print("Note: H2 uses PerformanceNASA as a proxy for objective task performance, with HRV_RMSSD as covariate.")
        else:
            print("Skipping H2: 'PerformanceNASA' or 'HRV_RMSSD' column not found in df_analysis.")

        # H3: Perceived Immersion, with HRV_RMSSD
        if 'PerceivedImmersion' in df_analysis.columns and 'HRV_RMSSD' in df_analysis.columns:
            formula_h3_cov = "PerceivedImmersion ~ ImmersionLevel + HRV_RMSSD"
            lmm_results['H3_PerceivedImmersion_HRV'] = fit_lmm(formula_h3_cov, df_analysis, dependent_var_name='PerceivedImmersion')
        else:
            print("Skipping H3: 'PerceivedImmersion' or 'HRV_RMSSD' column not found in df_analysis.")

        # H4: Cognitive Load (NASA-TLX subscales), with HRV_RMSSD
        nasa_tlx_subscales_for_h4 = {
            'MentalDemand': 'Mental Demand',
            'PhysicalDemand': 'Physical Demand',
            'TemporalDemand': 'Temporal Demand',
            'EffortNASA': 'Effort',
            'FrustrationNASA': 'Frustration'
        }
        print("\n--- H4: Cognitive Load (NASA-TLX Subscales) with HRV_RMSSD ---")
        for col_name, readable_name in nasa_tlx_subscales_for_h4.items():
            if col_name in df_analysis.columns and 'HRV_RMSSD' in df_analysis.columns:
                formula_h4_cov = f"{col_name} ~ ImmersionLevel + HRV_RMSSD"
                lmm_results[f'H4_{col_name}_HRV'] = fit_lmm(formula_h4_cov, df_analysis, dependent_var_name=col_name)
            else:
                print(f"Skipping H4 subscale {readable_name} ('{col_name}'): column or HRV_RMSSD not found in df_analysis.")
        
        # Store df_analysis for post-hoc and assumption checking cells
        %store df_analysis
        %store lmm_results

    else:
        print("Skipping Hypothesis Testing as df_analysis is empty after merging or initial loading.")


Successfully loaded df_post_test_cleaned and final_df (HRV data).
Shape of df_post_test_cleaned: (45, 48)
Shape of final_df (HRV): (75, 94)

Head of final_df (HRV data):
  ParticipantID Condition     Phase  HRV_MeanNN     HRV_SDNN  HRV_SDANN1  \
0            10  MediumFi  Baseline  660.764801   135.314503   31.175987   
1            10  MediumFi      Task  661.646606   815.397120  163.615276   
2             1    HighFi  Baseline  715.679284   226.257083   48.481917   
3             1    HighFi      Task  727.953191  1189.775585  273.490521   
4             2    HighFi  Baseline  565.038654    57.817954   16.013175   

   HRV_SDNNI1  HRV_SDANN2  HRV_SDNNI2  HRV_SDANN5  ...  HRV_SampEn  \
0  123.251099   17.586581  124.454787         NaN  ...    0.748622   
1  525.514743   87.186293  544.865326         NaN  ...    0.260511   
2  220.196165   40.496413  220.077430         NaN  ...    0.637484   
3  930.099612  110.209036  989.813782         NaN  ...    0.284032   
4   49.148874   19.7034

/tmp/ipykernel_104717/3403754238.py:96: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





               Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  PerceivedImmersion
No. Observations:  45       Method:              REML              
No. Groups:        44       Scale:               0.0010            
Min. group size:   1        Log-Likelihood:      -77.6876          
Max. group size:   2        Converged:           Yes               
Mean group size:   1.0                                             
-------------------------------------------------------------------
                         Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------------
Intercept                 7.265    0.244 29.835 0.000  6.788  7.742
ImmersionLevel[T.Medium] -0.001    0.045 -0.012 0.990 -0.089  0.088
ImmersionLevel[T.High]    1.037    0.383  2.708 0.007  0.287  1.787
HRV_RMSSD                -0.000    0.000 -0.236 0.813 -0.000  0.000
Group Var                 1.418  462.638                       

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning:

Maximum Likelihood optimization failed to converge. Check mle_retvals

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2206: ConvergenceWarning:

MixedLM optimization failed, trying a different optimizer may help.

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2218: ConvergenceWarning:

Gradient optimization failed, |grad| = 0.166177

/home/mitchell/Documents/Projects/P8-Project/.venv/lib64/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning:

The Hessian matrix at the estimated parameter values is not positive definite.



### 6. Post-hoc Analysis (if overall LMM/ANOVA is significant)

If the LMM for a hypothesis shows a significant overall effect of `ImmersionLevel`, we can perform pairwise comparisons to see which specific groups differ.
We can use `pingouin.mixed_anova` to get an ANOVA-like table from the LMM (though LMM summary already gives p-values for fixed effects) and `pingouin.pairwise_tests` for post-hoc tests.

In [ ]:
%store -r df_analysis # Load the DataFrame used for LMMs
%store -r lmm_results # Load the LMM results

if 'df_analysis' in locals() and not df_analysis.empty and 'lmm_results' in locals() and lmm_results:
    print("\n--- Post-hoc Analysis (Pingouin) ---")
    for model_name, result_object in lmm_results.items():
        if result_object is None:
            print(f"Skipping post-hoc for {model_name} as LMM fitting failed or was skipped.")
            continue

        dependent_var = result_object.model.endog_names
        # Check if ImmersionLevel was significant in the LMM.
        try:
            p_values_immersion = result_object.pvalues[[idx for idx in result_object.pvalues.index if 'ImmersionLevel' in idx]]
            is_significant_overall = (p_values_immersion < 0.05).any()
        except Exception as e:
            print(f"Could not directly assess significance of ImmersionLevel for {model_name} from p-values: {e}. Proceeding with caution for post-hoc.")
            is_significant_overall = (result_object.pvalues < 0.05).any() if hasattr(result_object, 'pvalues') else False

        if is_significant_overall:
            print(f"\nOverall effect of ImmersionLevel likely significant for {dependent_var} (model: {model_name}). Performing pairwise tests.")
            try:
                # Ensure the DV column for pairwise_tests is numeric
                df_analysis[dependent_var] = pd.to_numeric(df_analysis[dependent_var], errors='coerce')
                
                # Define necessary columns for this specific pairwise test
                # Include covariates if they were part of the model and pingouin supports them in this context, 
                # otherwise, pairwise tests are typically for the main factor of interest.
                # For pg.pairwise_tests with 'between' and 'subject', covariates are not directly handled in the same way as LMM.
                # The test will compare levels of 'ImmersionLevel' on 'dependent_var'.
                cols_for_pairwise = [dependent_var, 'ImmersionLevel', 'ParticipantID']
                # Add other covariates if the specific pairwise test function supports them or if you intend to subset/control manually.
                # For now, focusing on the primary factor ImmersionLevel.
                
                current_data_for_pairwise = df_analysis.dropna(subset=cols_for_pairwise)
                
                if current_data_for_pairwise.empty or current_data_for_pairwise[dependent_var].isnull().all():
                    print(f"No data or all NaNs for {dependent_var} for pairwise test after dropping NaNs from {cols_for_pairwise}. Skipping.")
                    continue
                if current_data_for_pairwise['ImmersionLevel'].nunique() < 2:
                    print(f"Not enough levels of ImmersionLevel for {dependent_var} for pairwise test. Skipping.")
                    continue

                pairwise_res = pg.pairwise_tests(data=current_data_for_pairwise, dv=dependent_var, 
                                                 between='ImmersionLevel', subject='ParticipantID', 
                                                 padjust='bonf') # Bonferroni correction
                print(f"\nPairwise comparisons for {dependent_var} (Bonferroni corrected):")
                print(pairwise_res)
            except Exception as e:
                print(f"Error during post-hoc analysis for {dependent_var} (model: {model_name}): {e}")
                print(f"Data for {dependent_var} (head):\n{df_analysis[cols_for_pairwise].head()}")
        else:
            print(f"\nOverall effect of ImmersionLevel not significant (or p-values unclear) for {dependent_var} (model: {model_name}). Skipping pairwise tests.")
else:
    print("Skipping Post-hoc Analysis as df_analysis is not available, empty, or no LMM results found.")


### 7. Check LMM Assumptions

For each fitted LMM, we should inspect residuals for:
1.  **Normality**: Q-Q plots, histograms, Shapiro-Wilk test.
2.  **Homoscedasticity**: Plot residuals vs. fitted values.

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

%store -r df_analysis # Load the DataFrame used for LMMs
%store -r lmm_results # Load the LMM results

if 'lmm_results' in locals() and lmm_results and 'df_analysis' in locals() and not df_analysis.empty:
    print("\n--- Checking LMM Assumptions ---")
    for model_name, result_object in lmm_results.items():
        if result_object is None:
            print(f"Skipping assumption checks for {model_name} as LMM fitting failed or was skipped.")
            continue
        
        dependent_var = result_object.model.endog_names
        print(f"\nChecking assumptions for model: {model_name} (DV: {dependent_var})")
        
        # Get residuals and fitted values
        residuals = result_object.resid
        fitted_values = result_object.fittedvalues

        if residuals is None or fitted_values is None or len(residuals) == 0:
            print(f"Could not retrieve residuals or fitted values for {model_name}. Skipping assumption checks.")
            continue
        
        # 1. Normality of Residuals
        print("  1. Normality of Residuals:")
        # Histogram
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        sns.histplot(residuals, kde=True)
        plt.title(f'Histogram of Residuals ({model_name})')
        
        # Q-Q Plot
        plt.subplot(1, 2, 2)
        stats.probplot(residuals, dist="norm", plot=plt)
        plt.title(f'Q-Q Plot of Residuals ({model_name})')
        plt.show()
        
        # Shapiro-Wilk Test
        if len(residuals) > 3: # Shapiro-Wilk needs at least 3 samples
            shapiro_stat, shapiro_p = stats.shapiro(residuals)
            print(f"    Shapiro-Wilk Test for Normality of Residuals: Statistic={shapiro_stat:.3f}, p-value={shapiro_p:.3f}")
            if shapiro_p < 0.05:
                print("    Note: Shapiro-Wilk test suggests residuals may not be normally distributed (p < 0.05).")
            else:
                print("    Shapiro-Wilk test suggests residuals are normally distributed (p >= 0.05).")
        else:
            print("    Skipping Shapiro-Wilk test: not enough residuals (need > 3).")

        # 2. Homoscedasticity (Constant Variance of Residuals)
        print("\n  2. Homoscedasticity of Residuals:")
        plt.figure()
        plt.scatter(fitted_values, residuals)
        plt.axhline(0, color='red', linestyle='--')
        plt.xlabel('Fitted Values')
        plt.ylabel('Residuals')
        plt.title(f'Residuals vs. Fitted Values ({model_name})')
        plt.show()
        print("    Inspect the plot above: Residuals should be randomly scattered around the horizontal line at 0, with no clear patterns (e.g., funnel shape).")

else:
    print("Skipping LMM Assumption Checks as lmm_results or df_analysis are not available or empty.")


### 8. Documentation and Interpretation

**Summary of Findings:**

*(This section should be filled in after reviewing the LMM outputs, post-hoc tests, and assumption checks.)*

*   **H1 & H1a (Performance Confidence):** 
    *   Interpret the coefficient for `ImmersionLevel`. Is there a significant effect? 
    *   If `UserSkill` was included, how did it affect performance confidence? Was the interaction significant?
*   **H2 (Task Performance - Proxy: PerceivedPerformanceRaw):**
    *   How did `ImmersionLevel` relate to perceived task performance? 
    *   Acknowledge the limitation of using a subjective proxy.
*   **H3 (Perceived Immersion):**
    *   Did `ImmersionLevel` significantly predict `PerceivedImmersion`? This serves as a manipulation check.
*   **H4 (Cognitive Load - NASA-TLX):**
    *   For each subscale (Mental Demand, Physical Demand, etc.), what was the effect of `ImmersionLevel`?

**LMM Interpretation Notes:**

*   **Coefficients:** For categorical predictors like `ImmersionLevel`, one level is treated as the reference (usually the first alphabetically or by defined order, e.g., 'LowFi'). Coefficients for other levels (e.g., `ImmersionLevel[T.MedFi]`, `ImmersionLevel[T.HighFi]`) represent the difference in the dependent variable compared to this reference level, holding other variables constant.
*   **P-values (P>|z|):** Indicate the statistical significance of each coefficient. A p-value < 0.05 typically suggests a significant effect.
*   **Confidence Intervals ([0.025 0.975]):** Provide a range of plausible values for the true coefficient. If the interval does not include 0, the coefficient is statistically significant at the 95% confidence level.
*   **Random Effects (Group Var):** The variance of `ParticipantID` indicates the extent of individual differences in the baseline level of the dependent variable.

**Limitations:**

*   **Sample Size:** Consider if the sample size is adequate for the complexity of the models and the number of groups.
*   **Proxy Variables:** If objective task performance data was unavailable, the use of `PerceivedPerformanceRaw` is a limitation.
*   **Missing Data:** How was missing data handled, and could it bias results? (e.g., if `UserSkill` data was largely missing).
*   **LMM Assumptions:** Were all assumptions of LMMs met? If not, how might this affect the conclusions? (e.g., non-normal residuals might affect p-value accuracy).
*   **Multiple Comparisons:** If many post-hoc tests were run, was an appropriate correction (e.g., Bonferroni) used? This was included in the `pingouin.pairwise_tests` example.

*(Further detailed interpretation based on specific model outputs will be added here once the models are run and results are available.)*